# Equipment Calibration for HI Lab: Part 1 of 3

This notebook is the **Part 1 (equipment-calibration) stage** in a three-part HI analysis chain:

1. **Part 1 (this notebook):** characterize attenuation, gain-scale linearity, reflectometry constraints, and hardware bandpass responses.
2. **Part 2 (`temperature_calibration.ipynb`):** apply intensity calibration to obtain physically scaled spectra.
3. **Part 3 (`analysis.ipynb`):** perform astrophysical line analysis (velocity-frame conversion, profile interpretation, and science products).

The goal here is to produce physically defensible calibration products with uncertainties and explicit requirement traceability to the lab handouts.

## Requirement Traceability (Part 1 Scope)

Primary references:

- `src/ugradio/lab_bighorn/bighorn.tex`
- `src/ugradio/lab_bighorn/cal_intensity.tex`
- `src/ugradio/lab_bighorn/horn_signal_chain_test.tex`
- `src/ugradio/lab_bighorn/coords.tex`

| ID | Requirement / theory item | Where addressed | Status |
|---|---|---|---|
| `R-SC-001` | Receiver chain gain/loss accounting (horn -> SDR) | Signal-chain gain sections + bench chain metrics | Covered |
| `R-SC-002` | Coax attenuation per meter near 1420 MHz | Shared-slope SDR + meter linear fits | Covered |
| `R-SC-003` | Unknown cable length inference with uncertainty | Inversion + Jacobian propagation | Covered |
| `R-SC-004` | SDR gain-scale linearity and clipping guardrails | Fixed-gain sweep diagnostics | Covered |
| `R-SC-005` | RTL2832U FIR + residual summing-filter response | FIR + cold_ref whitening + constrained optimization | Covered |
| `R-SC-006` | Reflectometry timing ambiguity and branch selection | Square-wave branch resolution and velocity-factor check | Covered |
| `R-SC-007` | VSWR/reflection-coefficient requirement awareness | Explicitly documented as a measurement gap | Covered with limitation |
| `R-CAL-001` | `cal_intensity.tex` cool-method calibration theory | Full derivation block below | Covered |
| `R-CAL-002` | First-order error propagation (sum/product/Jacobian) | Uncertainty derivation + implementation mapping | Covered |
| `R-COORD-001` | Coordinate transforms for sky/LSR analysis | Deferred to `analysis.ipynb` (Part 3), linked in handoff | Deferred intentionally |

## 1) Physical Setup and Modeling Assumptions

This section defines constants and assumptions used throughout Part 1.

### Hardware and measurement assumptions

- Coax attenuation over the sampled length range is modeled as first-order linear in dB: `y(L)=B-\alpha L`.
- Splitter and fixed attenuator losses are treated as additive constants in dB (intercept shifts), not slope changes.
- The 12-ft lead-in (`3.6576 m`) is subtracted only at the final unknown-cable estimate stage.

### Statistical assumptions

- Fit-parameter uncertainty combines OLS covariance with Monte Carlo perturbation from cable-length read uncertainty.
- Length inversion uncertainty is propagated by first-order Jacobian rules (consistent with `cal_intensity.tex`).

### Scope boundary

This notebook calibrates hardware terms used downstream; it does not perform sky-coordinate transforms or final astrophysical HI interpretation.

### Measurement-resolution inputs used for uncertainty propagation

- Ruler smallest division: `0.2 mm` -> per-read standard uncertainty proxy `sigma_L_read = 0.1 mm`.
- Power-meter smallest division: `0.2 dBm` -> per-read standard uncertainty proxy `sigma_P_read = 0.1 dB`.


## 2) Bench-Measured Signal-Chain Gain Context

The cumulative-gain plot uses **bench-probed block values** (dated 2026-03-07) to summarize end-to-end gain/loss structure from horn-side chain to SDR input.

Interpretation focus:

- verify that measured gain budget is plausible for ADC usage,
- identify anomalous losses (notably unexpectedly high K&L insertion loss),
- establish physically consistent priors for later calibration and safety checks.

## 3) Analysis Operators and Theory-to-Code Mapping

Helper functions below implement the mathematical operators used in this notebook.

### Model equations implemented

- Shared-slope SDR attenuation model:
  $$
  y_{1420}(L)=B_{1420}-\alpha L,\qquad y_{1421}(L)=B_{1421}-\alpha L
  $$
- Meter branch model:
  $$
  y_{\mathrm{meter}}(L)=B_{\mathrm{meter}}-\alpha_{\mathrm{meter}}L
  $$
- Unknown-length inversion:
  $$
  L_i=\frac{B_i-y_i}{\alpha},\quad i\in\{1420,1421\}
  $$

### Error-propagation operators implemented

- `propagate_length_sigma(...)` for first-order Jacobian propagation,
- shared-fit covariance use for coupled parameters `[B_{1420}, B_{1421}, \alpha]`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

LAB02_DIR = Path.cwd().resolve()
if not (LAB02_DIR / "utils").exists():
    LAB02_DIR = (Path.cwd() / "labs" / "02").resolve()
if str(LAB02_DIR) not in sys.path:
    sys.path.insert(0, str(LAB02_DIR))

pd.options.display.max_columns = 200
pd.options.display.float_format = "{:.4f}".format

from utils import run_equipment_calibration
from utils.plotting import figure
from utils.tables import table

## 4) Full `cal_intensity.tex` Cool-Method Derivation (Theory Alignment)

We reproduce the calibration logic used for spectral-line intensity scaling.

Start with the three measured spectra (channel index `j`):

$$
P_j^{ONLINE,CALOFF}=G_j\left(T_{sys}+T_{ant,HI}(\nu)\right),
$$
$$
P_j^{OFFLINE,CALOFF}=G_jT_{sys},
$$
$$
P_j^{OFFLINE,CALON}=G_j\left(T_{sys}+T_{cal}\right).
$$

From the first two equations,

$$
T_{sys}+T_{ant,HI}(\nu)=\left[\frac{P_j^{ONLINE,CALOFF}}{P_j^{OFFLINE,CALOFF}}\right]T_{sys}.
$$

The cool-method estimate of `T_sys` is obtained by band-averaging the noisy cal-difference ratio:

$$
T_{sys}=\frac{\sum_j P_j^{OFFLINE,CALOFF}}
{\sum_j\left(P_j^{OFFLINE,CALON}-P_j^{OFFLINE,CALOFF}\right)}\,T_{cal}.
$$

Substitution yields the calibrated spectrum estimate:

$$
T_{sys}+T_{ant,HI}(\nu)=
\left[\frac{P_j^{ONLINE,CALOFF}}{P_j^{OFFLINE,CALOFF}}\right]
\left[
\frac{\sum_j P_j^{OFFLINE,CALOFF}}
{\sum_j\left(P_j^{OFFLINE,CALON}-P_j^{OFFLINE,CALOFF}\right)}
\right]T_{cal}.
$$

Why cool beats naive: the channel-by-channel noisy denominator is replaced by a high-S/N band-averaged scale factor, reducing channel-scale noise while preserving line shape.

Part 1 note: this notebook focuses on hardware characterization terms that feed this calibration chain; full sky-line intensity products are produced in Part 2.

## 5) Data Ingest, Normalization, and QC Policy

We load attenuation and unknown-length manifests, validate required fields, normalize by source setpoint, and apply diagnostics-first screening.

Normalization used throughout:

$$
y = 10\log_{10}(P_{\mathrm{total}})-P_{\mathrm{siggen,dBm}}.
$$

This removes intentional source-level changes so slope estimates isolate cable-length dependence.

In [ ]:
equipment = run_equipment_calibration()
display(Markdown(
    f"Part 1 produced `{len(equipment.tables)}` tables and `{len(equipment.figures)}` figures. "
    f"Artifact: `{equipment.artifact_path}`"
))

In [ ]:
for name in ["attenuation_primary", "unknown_manifest", "screening_diagnostics"]:
    display(Markdown(f"### {name.replace('_', ' ').title()}"))
    display(table(equipment, name))

## 6) Path Corrections and Intercept Accounting

Known splitter/attenuator losses are applied as fixed dB offsets.

- These corrections shift intercept terms (`B`) but do not change `\alpha`.
- The short port-2 cable correction is deferred until `\alpha` is estimated.
- Unknown-length inversion remains invariant when the same fixed branch corrections apply consistently to calibration and unknown datasets.

## 7) Shared-Slope Attenuation Fit (Primary SDR Model)

We fit all points, perform residual and leave-one-out influence diagnostics, and select the primary fit according to the screening policy.

Core model:

$$
y_i(L)=B_i-\alpha L,\quad i\in\{1420,1421\}.
$$

### SDR Fit Interpretation

Interpretation targets:

- Is a single physically meaningful slope `\alpha` supported across LO branches?
- Do residuals show random scatter around zero without systematic curvature?
- Do screened and all-point fits differ in a way consistent with leverage/outlier diagnostics?

## 8) Uncertainty Propagation (First-Order, `cal_intensity`-style)

We use first-order propagation with covariance terms retained.

For
$$
L=\frac{B-y}{\alpha},
$$

Jacobian terms are
$$
\frac{\partial L}{\partial B}=\frac{1}{\alpha},\qquad
\frac{\partial L}{\partial y}=-\frac{1}{\alpha},\qquad
\frac{\partial L}{\partial \alpha}=-\frac{L}{\alpha}.
$$

Variance (including covariance between `B` and `\alpha`):
$$
\sigma_L^2=
\left(\frac{\partial L}{\partial B}\right)^2\sigma_B^2+
\left(\frac{\partial L}{\partial y}\right)^2\sigma_y^2+
\left(\frac{\partial L}{\partial \alpha}\right)^2\sigma_\alpha^2+
2\frac{\partial L}{\partial B}\frac{\partial L}{\partial \alpha}\operatorname{Cov}(B,\alpha).
$$

Shared-SDR unknown-length uncertainty additionally uses the full covariance on `[B_{1420}, B_{1421}, \alpha]`.

## 9) Model Scope and Why We Keep a Linear Attenuation Law

We intentionally keep the attenuation model linear in dB vs length for this dataset size.

- Sparse length sampling makes higher-order fits unstable.
- Unknown-length inversion is highly sensitive to slope uncertainty.
- Diagnostics are used to handle leverage/outlier behavior instead of adding unnecessary model degrees of freedom.

## 10) Independent Meter-Branch Cross-Check

We fit the meter branch independently and compare attenuation slopes with SDR-based estimates.

Agreement supports a path-independent physical attenuation interpretation; disagreement indicates branch-specific systematic effects (termination mismatch, readout offsets, connector state, or compression).

## 11) Voltage-Space Cross-Check (50 Ohm Matched Assumption)

We convert power-meter dBm readings to equivalent `V_rms` and `V_pp` under a matched 50 Ohm load.

This is an internal sanity check only, not the primary calibration path.

## 12) Bench Analog Measurement Log (2026-03-07)

This section documents the measured A1-A5 chain configurations and derives block-level gain/loss metrics.

It provides:

- chain closure checks,
- unknown-cable bench estimate for consistency,
- conservative drive-level safety guidance for cascaded amplifier tests.

### Filter/Amplifier Ordering and Noise Implications

Ordering affects both gain distribution and total in-band/out-of-band noise delivered to the ADC.

Following the logic in `horn_signal_chain_test.tex` and `bighorn.tex`, the objective is to preserve enough gain for robust digitization while constraining out-of-band amplification and reflection-induced artifacts.

## 13) Unknown-Cable Inversion and Primary-Estimate Selection

Unknown-length estimates:

$$
L_{1420}=\frac{B_{1420}-y_{1420}^{obs}}{\alpha},\quad
L_{1421}=\frac{B_{1421}-y_{1421}^{obs}}{\alpha},
$$
$$
L_{SDR}=\frac{L_{1420}+L_{1421}}{2},\quad
L_{meter}=\frac{B_{meter}-y_{meter}^{obs}}{\alpha_{meter}},
$$
$$
L_{unknown}=L_{total}-L_{lead}.
$$

Primary-source decision is criterion-based (SDR first; meter fallback if SDR stability criteria fail).

### Unknown-Length Cross-Checks

We compare manifest SDR, manifest meter, and bench analog estimates to verify calibration closure and to expose systematic offsets before exporting downstream calibration products.

## 14) Reflectometry Branch Resolution for Cable Velocity

Square-wave reflectometry provides round-trip delay constraints; periodic ambiguity is resolved by branch selection:

$$
\Delta t_n=\Delta t_{mod}+nT.
$$

Using unknown-cable length priors from the manifest inversion, we compute

$$
v_{coax}=\frac{2L}{\Delta t_n}
$$

and select physically plausible branches against expected velocity factor.

### VSWR / Reflection-Coefficient Requirement Status

`bighorn.tex` asks for VSWR/reflection quantification. This notebook includes timing-based reflectometry for propagation speed, but does not directly measure `|\Gamma|` or VSWR because incident/reflected amplitudes were not separately instrumented (no directional coupler/VNA in this run).

Resulting impact: mismatch effects may remain folded into empirical attenuation terms.

## 15) Consolidated Part-1 Summary Tables

Summary tables collect fit quality, attenuation products, unknown-length estimates with uncertainties, and selection rationale for the exported primary estimate.

In [ ]:
for name in [
    "fit_compare",
    "estimate_summary",
    "fit_quality",
    "meter_compare",
    "meter_voltage",
    "unknown_crosscheck",
    "reflectometry_branches",
]:
    display(Markdown(f"### {name.replace('_', ' ').title()}"))
    display(table(equipment, name))

### Physical Plausibility Against Coax Expectations

Measured attenuation slopes are compared against RG-58-family expectations near 1420 MHz to test whether fitted values are physically credible and to flag mechanically compromised calibration points.

### Additional Consistency Diagnostics

This section records agreement/disagreement patterns across screened vs all-point fits, manifest vs bench estimates, and safety/compression constraints used for conservative bench operation.

## 16) SDR Gain-Sweep Linearity and Clipping Guardrails

We analyze fixed-gain sweep data to identify the usable linear-response range and clipping thresholds that constrain reliable calibration operation.

In [ ]:
display(Markdown("### Sweep Fit"))
display(table(equipment, "sweep_fit"))
display(figure(equipment, "sdr_gain_response_clipping"))

## 17) Part-2 / Part-3 Handoff and Scope Boundary

### Outputs consumed downstream

- attenuation and unknown-length calibration terms,
- linearity guardrails,
- FIR/summing response arrays and evaluation masks.

### Coordinate-theory boundary

`coords.tex` implementation and sky-frame transforms are intentionally handled in `analysis.ipynb` (Part 3), not in this equipment notebook.

## 18) RTL2832U FIR and Summing-Filter Response Modeling

Using `bighorn.tex` default FIR coefficients and the estimated secondary summing filter, we model response on the measured output-frequency grid and define passband support masks for robust correction.

In [ ]:
display(Markdown("### Response Summary"))
display(table(equipment, "response_summary"))
for name in [
    "signal_chain",
    "cable_attenuation_lo",
    "cable_attenuation_power_meter",
    "reflectometry",
    "sdr_fir_summing_correction",
]:
    display(Markdown(f"### {name.replace('_', ' ').title()}"))
    display(figure(equipment, name))

## 19) White-Noise Verification Using `cold_ref`

We use combined cold-reference spectra as a broadband proxy and evaluate whether model-based correction flattens the passband over response-supported channels.

## 20) Constrained Summing-Filter Optimization

The residual passband ripple is reduced by fitting a symmetric 11-tap summing filter (`[a,b,c,d,e,f,e,d,c,b,a]`) to minimize variance of corrected broadband power in the evaluation mask.

## 21) Export Contract (Hard Cutover to v2)

This notebook exports `equipment_calibration_results_v2.npz` under `labs/02/cache/` with `schema_version = "2.0.0"`.

The v2 artifact is grouped by semantic namespace (`model.*`, `length.*`, `response.*`, `linearity.*`, `provenance.*`, `trace.*`) and is intended to be the sole downstream contract for Part 2.

In [ ]:
display(Markdown("### Bench Metrics"))
display(table(equipment, "bench_metrics"))
display(Markdown("### Bench Log"))
display(table(equipment, "bench_log"))
display(Markdown(f"Saved `{len(equipment.artifact)}` artifact fields to `{equipment.artifact_path}`."))